In [1]:
import torch
from torch import nn
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader
from torchvision.transforms import v2

In [2]:
train_data = FashionMNIST(
    root="../Datasets",
    download=False,
    train=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(dtype=torch.float32, scale=True)])
)

In [3]:
test_data = FashionMNIST(
    root="../Datasets",
    download=False,
    train=False,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(dtype=torch.float32, scale=True)])
)

In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.flatten = nn.Flatten()
        self.linear_stack = nn.Sequential(
            nn.Linear(in_features=28*28, out_features=512),
            nn.ReLU(),
            nn.Linear(in_features=512, out_features=512),
            nn.ReLU(),
            nn.Linear(in_features=512, out_features=10)
        )
        
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_stack(x)
        return logits

In [5]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

In [6]:
train_loader = DataLoader(dataset=train_data, batch_size=batch_size)
test_loader = DataLoader(dataset=test_data, batch_size=batch_size)

In [7]:
for batch, (x,y) in enumerate(train_loader):
    print(y)
    break

tensor([9, 0, 0, 3, 0, 2, 7, 2, 5, 5, 0, 9, 5, 5, 7, 9, 1, 0, 6, 4, 3, 1, 4, 8,
        4, 3, 0, 2, 4, 4, 5, 3, 6, 6, 0, 8, 5, 2, 1, 6, 6, 7, 9, 5, 9, 2, 7, 3,
        0, 3, 3, 3, 7, 2, 2, 6, 6, 8, 3, 3, 5, 0, 5, 5])


In [8]:
model = NeuralNetwork()
optimizer = torch.optim.SGD(params = model.parameters(), lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()

In [9]:
def train_loop(model, optimizer, train_loader, loss_fn):
    losses = []
    n = len(train_loader.dataset)
    model.train()
    for batch, (x,y) in enumerate(train_loader):
        y_pred = model(x)
        #print(y_pred.shape,y.shape)
        loss = loss_fn(y_pred, y)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        losses.append(loss.item())
        if batch%100 == 0:
            current = batch * train_loader.batch_size + len(x)
            print(f"loss: {loss:>7f} [{current:>7d}/{n}]")
        
    return losses

In [10]:
def test_loop(test_loader, model):
    model.eval()
    n = len(test_loader.dataset)
    num_batches = len(test_loader)
    print(num_batches)
    correct, loss = 0, 0.0
    with torch.no_grad():
        for batch, (x,y) in enumerate(test_loader):
            y_pred = model(x)
            loss += loss_fn(y_pred, y) * x.shape[0]
            #loss += loss_fn(y_pred, y)
            correct += (y_pred.argmax(dim=1) == y).type(torch.float).sum().item()
    print(f"Accuracy: {100*(correct/n):.3f}%, Loss: {(loss/n):.3f}")
test_loop(test_loader, model)

157
Accuracy: 11.860%, Loss: 2.303


In [11]:
epochs = 5
model = NeuralNetwork()
optimizer = torch.optim.SGD(params = model.parameters(), lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(epochs):
    print(f"Epoch: {epoch+1}/{epochs}")
    train_loop(model, optimizer, train_loader, loss_fn)
    test_loop(test_loader, model)
    print("\n\n")

Epoch: 1/5
loss: 2.305152 [     64/60000]
loss: 2.287098 [   6464/60000]
loss: 2.264232 [  12864/60000]
loss: 2.253245 [  19264/60000]
loss: 2.256530 [  25664/60000]
loss: 2.213694 [  32064/60000]
loss: 2.225546 [  38464/60000]
loss: 2.190195 [  44864/60000]
loss: 2.193521 [  51264/60000]
loss: 2.158201 [  57664/60000]
157
Accuracy: 44.200%, Loss: 2.149



Epoch: 2/5
loss: 2.167416 [     64/60000]
loss: 2.151738 [   6464/60000]
loss: 2.093684 [  12864/60000]
loss: 2.106688 [  19264/60000]
loss: 2.070174 [  25664/60000]
loss: 2.000410 [  32064/60000]
loss: 2.031463 [  38464/60000]
loss: 1.948917 [  44864/60000]
loss: 1.966207 [  51264/60000]
loss: 1.891131 [  57664/60000]
157
Accuracy: 57.270%, Loss: 1.885



Epoch: 3/5
loss: 1.923895 [     64/60000]
loss: 1.888609 [   6464/60000]
loss: 1.773731 [  12864/60000]
loss: 1.811581 [  19264/60000]
loss: 1.710100 [  25664/60000]
loss: 1.651452 [  32064/60000]
loss: 1.675028 [  38464/60000]
loss: 1.567514 [  44864/60000]
loss: 1.606465 [  51264

In [13]:
model

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)

In [14]:
for key, value in model.state_dict().items():
    print(f"{key=}, weights shape:{value.shape}")

key='linear_stack.0.weight', weights shape:torch.Size([512, 784])
key='linear_stack.0.bias', weights shape:torch.Size([512])
key='linear_stack.2.weight', weights shape:torch.Size([512, 512])
key='linear_stack.2.bias', weights shape:torch.Size([512])
key='linear_stack.4.weight', weights shape:torch.Size([10, 512])
key='linear_stack.4.bias', weights shape:torch.Size([10])
